# 00 — Cek Lingkungan

Jalankan notebook ini **pertama kali** di setiap mesin (mis. PC rumah dengan RTX 2060)
sebelum menjalankan notebook 01–05.

**Yang dicek:** versi Python/OS, paket Python (torch + CUDA, sentence-transformers,
chromadb, dll.), GPU & VRAM, serta koneksi ke **Ollama** dan model embedding.

> Semua blok memakai `try/except`, jadi notebook tidak berhenti walau ada yang belum
> terpasang — kita hanya ingin tahu apa yang kurang.

## 1. Versi Python, OS, dan paket Python

In [ ]:
import platform
import sys

print("Python    :", sys.version.split()[0])
print("OS        :", platform.platform())
print("Arsitektur:", platform.machine())

packages = [
    "numpy", "pandas", "sklearn", "pypdf", "psutil",
    "torch", "sentence_transformers", "chromadb",
    "langchain_text_splitters", "datasets", "ragas", "ollama",
]

status_paket = {}
print("\nPaket Python:")
for name in packages:
    try:
        mod = __import__(name)
        status_paket[name] = getattr(mod, "__version__", "terpasang")
    except Exception as e:
        status_paket[name] = f"BELUM ({type(e).__name__})"
    print(f"  {name:26s}: {status_paket[name]}")

## 2. GPU & CUDA (khusus RTX 2060)

Di PC rumah, `torch.cuda.is_available()` harus `True` dan VRAM terbaca ±6 GB.

In [ ]:
import shutil
import subprocess

try:
    import torch
    print("Torch         :", torch.__version__)
    print("CUDA tersedia :", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU           :", torch.cuda.get_device_name(0))
        print("VRAM total    :", round(props.total_memory / 1024**3, 1), "GB")
        print("Compute cap.  :", f"sm_{props.major}{props.minor}")
    else:
        print("-> Jalan di CPU. Untuk RTX, pasang torch CUDA (lihat README Langkah 2).")
except Exception as e:
    print("Torch belum siap:", type(e).__name__, "-", e)

print("\n-- nvidia-smi --")
if shutil.which("nvidia-smi"):
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(out.stdout or out.stderr)
else:
    print("nvidia-smi tidak ditemukan (tidak ada GPU NVIDIA / driver belum terpasang).")

## 3. Koneksi ke Ollama (Local LLM)

Butuh server Ollama **berjalan** dan model sudah diunduh:
`ollama pull llama3.1:8b-instruct-q4_K_M`.

In [ ]:
LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"

print("-- ollama list --")
if shutil.which("ollama"):
    out = subprocess.run(["ollama", "list"], capture_output=True, text=True)
    print(out.stdout or out.stderr)
else:
    print("Perintah 'ollama' tidak ditemukan di PATH.")

try:
    import ollama
    resp = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Balas satu kata: siap"}],
    )
    print("Respons LLM :", resp["message"]["content"].strip())
except Exception as e:
    print("Ollama belum siap:", type(e).__name__, "-", e)
    print(f"Pastikan server berjalan dan model '{LLM_MODEL}' sudah diunduh.")

## 4. Uji model embedding

Pertama kali dijalankan, model akan diunduh otomatis (±2,2 GB untuk `bge-m3`).

In [ ]:
EMBED_MODEL = "BAAI/bge-m3"

try:
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(EMBED_MODEL)
    vec = m.encode(["uji coba embedding"], normalize_embeddings=True)
    print("Embedding OK :", EMBED_MODEL)
    print("Dimensi      :", vec.shape[1])
except Exception as e:
    print("Embedding belum siap:", type(e).__name__, "-", e)
    print(f"Model '{EMBED_MODEL}' akan diunduh saat pertama dipakai.")

## 5. Ringkasan kesiapan

In [ ]:
def cek(ok, label):
    tanda = "[ SIAP ]" if ok else "[  --  ]"
    print(f"{tanda} {label}")

torch_ok = "torch" in status_paket and not str(status_paket["torch"]).startswith("BELUM")
cuda_ok = False
if torch_ok:
    import torch
    cuda_ok = torch.cuda.is_available()

print("=== RINGKASAN KESIAPAN ===")
cek(not str(status_paket.get("sentence_transformers", "BELUM")).startswith("BELUM"),
    "sentence-transformers terpasang")
cek(not str(status_paket.get("chromadb", "BELUM")).startswith("BELUM"),
    "chromadb terpasang")
cek(torch_ok, "PyTorch terpasang")
cek(cuda_ok, "GPU CUDA terdeteksi (untuk RTX 2060)")
cek(not str(status_paket.get("ollama", "BELUM")).startswith("BELUM"),
    "klien ollama terpasang")
cek(not str(status_paket.get("ragas", "BELUM")).startswith("BELUM"),
    "ragas terpasang (opsional, untuk notebook 03)")
cek(not str(status_paket.get("langchain_text_splitters", "BELUM")).startswith("BELUM"),
    "langchain-text-splitters terpasang (untuk notebook 02)")

print("\nBila ada yang '[  --  ]', pasang dulu lalu jalankan ulang sel ini.")
print("Contoh: pip install -r requirements.txt ; pip install torch --index-url https://download.pytorch.org/whl/cu121")